# Frames index

Dumps the headers of all raw frames of the given nights (all telescopes in `TELS`, all frame types: zeros, darks,
flats, science, ...) to `~/data/ocm_frames/<tel>/<night>.parquet`: one row per frame, one column per header card,
plus `file` (full path) and `night` (`int(JD) - 2460000`). Other notebooks read these files instead of opening every
raw file (`calib_stack`, `bias_qc`, `dark_qc` in `calibration_check/`, the `ghost/` work, ...).

- Empty cards (`''`) are stored as missing values. A card whose values are of different types within one night (e.g.
  `UOBI` as a number and as text) is stored as text in that night's file. `COMMENT` / `HISTORY` are not kept.
- Nights already in the index are skipped, unless `OVERWRITE = True`. The newest night may still be growing: re-run it
  with `OVERWRITE = True` (or delete its file) to get the frames taken later.
- `RAW_ROOT`: `/work/vela/oca/fits` on mizar (Warsaw), `/data/fits` on the observatory machines.

In [20]:
import os
import glob
import pandas as pd
from astropy.io import fits
from tqdm.notebook import tqdm

#TELS = ['wk06', 'zb08', 'jk15']
TELS = ['zb08',]

N_START = 1340     # first (oldest) night, int(JD) - 2460000
N_STOP = 1368      # last (newest) night
OVERWRITE = False  # True: read the nights again even if they are already in the index

RAW_ROOT = '/work/vela/oca/fits'
FRAMES_ROOT = os.path.expanduser('~/data/ocm_frames')

## Read the headers

In [21]:
for tel in TELS:
    os.makedirs(f'{FRAMES_ROOT}/{tel}', exist_ok=True)
    for night in tqdm(range(N_START, N_STOP + 1), desc=tel):
        out = f'{FRAMES_ROOT}/{tel}/{night}.parquet'
        if os.path.exists(out) and not OVERWRITE:
            print(tel, night, 'already in the index')
            continue
        files = sorted(glob.glob(f'{RAW_ROOT}/{tel}/raw/{night}/*.fits'))
        if not files:
            print(tel, night, 'no frames')
            continue

        rows = []
        for f in tqdm(files, desc=f'{tel} {night}', leave=False):
            h = fits.getheader(f)
            row = {'file': f, 'night': night}
            for k, v in h.items():
                if k in ('COMMENT', 'HISTORY', ''):
                    continue
                row[k] = None if v == '' or isinstance(v, fits.card.Undefined) else v
            rows.append(row)
        frames = pd.DataFrame(rows)

        # a column must hold one type: a card with mixed types (other than int + float) becomes text
        for k in frames.columns:
            kinds = {type(v) for v in frames[k] if v is not None and v == v}
            if len(kinds) > 1 and kinds != {int, float}:
                frames[k] = [None if v is None or v != v else str(v) for v in frames[k]]

        frames.to_parquet(out)
        print(tel, night, len(frames), 'frames:', ', '.join(f'{n} {t}' for t, n in frames['IMAGETYP'].value_counts().items()))

zb08:   0%|          | 0/29 [00:00<?, ?it/s]

zb08 1340 already in the index
zb08 1341 already in the index
zb08 1342 already in the index
zb08 1343 already in the index
zb08 1344 already in the index
zb08 1345 already in the index
zb08 1346 already in the index
zb08 1347 already in the index
zb08 1348 already in the index
zb08 1349 already in the index
zb08 1350 already in the index


zb08 1351:   0%|          | 0/918 [00:00<?, ?it/s]

zb08 1351 918 frames: 816 science, 80 flat, 22 zero


zb08 1352:   0%|          | 0/1162 [00:00<?, ?it/s]

zb08 1352 1162 frames: 1056 science, 84 flat, 22 zero


... (18 outputs left out)

zb08 1363:   0%|          | 0/727 [00:00<?, ?it/s]

zb08 1363 727 frames: 656 science, 49 flat, 22 zero


zb08 1364:   0%|          | 0/1158 [00:00<?, ?it/s]

zb08 1364 1158 frames: 1091 science, 44 flat, 22 zero, 1 domeflat
zb08 1365 already in the index
zb08 1366 already in the index
zb08 1367 already in the index
zb08 1368 already in the index


## Read the index back

How other notebooks read it: all nights of one telescope, one table (columns missing in some nights are filled with
missing values).

In [31]:
path = "/home/mgorski/data/ocm_frames/zb08/"
file = "1368.parquet"
fits = pd.read_parquet(path+file, engine='pyarrow')

print(len(fits))

print(fits.keys())

#for i,t in fits.iterrows():
#    print(t["file"], t["IMAGETYP"], t["OBSTYPE"])

tmp = fits.to_dict("records")

for t in tmp[:5]:
    print(t)


1073
Index(['file', 'night', 'SIMPLE', 'BITPIX', 'NAXIS', 'NAXIS1', 'NAXIS2',
       'OCASTD', 'OBSERVAT', 'OBS-LAT', 'OBS-LONG', 'OBS-ELEV', 'ORIGIN',
       'TELESCOP', 'PI', 'SCIPROG', 'OBS-ID', 'UOBI', 'DATE-OBS', 'JD', 'RA',
       'DEC', 'EQUINOX', 'RA_OBJ', 'DEC_OBJ', 'RA_TEL', 'DEC_TEL', 'ALT_TEL',
       'AZ_TEL', 'AIRMASS', 'OBSMODE', 'OBSERVER', 'IMAGETYP', 'OBSTYPE',
       'OBJECT', 'NLOOPS', 'LOOP', 'FILTER', 'EXPTIME', 'INSTRUME', 'T-CAM',
       'T-CAMSET', 'XBINNING', 'YBINNING', 'READ-MOD', 'GAIN-MOD', 'GAIN',
       'RON', 'SUBRASTR', 'SCALE', 'SATURATE', 'FLAT_ERA', 'ZERO_ERA',
       'DARK_ERA', 'CCD-BLCL', 'CCD-SCMP', 'CCD-PORT', 'CCD-VSSP', 'FOCUS',
       'ROTATOR', 'ROT-MECH', 'PIERSIDE', 'TRACKING', 'GUIDING', 'DOME-AZ',
       'DOME-SHT', 'MIRR-COV', 'FAN-MIRR', 'M1-POS1', 'M1-POS2', 'M1-POS3',
       'M1-ATSET', 'FAN-DOME', 'LAMP-FLT', 'P-WS', 'RHUM-WS', 'T-WS',
       'WIND-DIR', 'WIND-AVG', 'WIND-GUS', 'T-DOME', 'RHUM-DOM', 'TEST',
       'BZERO'],
      d

## Working with the table (pandas DataFrame)

`pd.read_parquet` gives a `DataFrame`: rows = frames, columns = header cards (+ `file`, `night`). A column is a
`Series` (values + index); a row from `iloc` / `iterrows()` is a `Series` too, labelled by the column names.

In [ ]:
import os

frames = pd.read_parquet(os.path.expanduser('~/data/ocm_frames/zb08/1368.parquet'))

# the table as a whole
print(frames.shape)              # (rows, columns)
print(frames.columns[:10])       # column names (same as frames.keys())
print(frames.dtypes['EXPTIME'], frames.dtypes['FILTER'])   # type of a column
display(frames.head(3))          # first rows

# one column = Series
print(frames['EXPTIME'].max())             # longest exposure
print(frames['FILTER'].unique())           # filters used this night
print(frames['IMAGETYP'].value_counts())   # number of frames of each type
print(frames['ROT-MECH'].describe())       # min, max, mean, ...

# several columns = a smaller DataFrame
display(frames[['file', 'FILTER', 'EXPTIME']].head(3))

# one row = Series, its labels are the column names
t = frames.iloc[0]
print(t['file'], t['FILTER'], t['EXPTIME'])

# selecting rows with a condition (& = and, | = or, each condition in brackets)
zf = frames[(frames['FILTER'] == 'z') & (frames['IMAGETYP'] == 'flat')]
print(len(zf), 'z flats')

# loop over the frames: one row = one file
for i, t in zf.iterrows():
    print(os.path.basename(t['file']), t['DATE-OBS'], t['EXPTIME'], round(t['ROT-MECH'], 1))

# missing values: NaN in numbers, None in text; check with pd.isna (not == None)
print(pd.isna(frames['SATURATE']).sum(), 'frames without SATURATE')

# to numpy / astropy if needed
exptime = frames['EXPTIME'].to_numpy()     # numpy array
# from astropy.table import Table; tab = Table.from_pandas(frames)